# Tahap 3 - Case Retrieval
Notebook ini hanya untuk Tahap 3 sesuai instruksi tugas.

Mencakup TF-IDF, SVM, Naive Bayes, IndoBERT, Recall@K, Precision@K, dan evaluasi retrieval.

In [1]:
!pip -q install transformers torch scikit-learn sentencepiece

In [ ]:
import pandas as pd, numpy as np, json, os
from google.colab import drive

# 1. Sambungkan ke Google Drive
drive.mount('/content/drive')

# 2. Sesuaikan path ke file cases.csv Anda di Drive
# Berdasarkan informasi Anda, pathnya kemungkinan ada di subfolder data/processed di dalam PDF_FOLDER
PDF_FOLDER = '/content/drive/My Drive/TugasPK/Test/raw_pdf'
csv_path = os.path.join(PDF_FOLDER, 'data/processed/cases.csv')

if os.path.exists(csv_path):
    cases = pd.read_csv(csv_path)
    print("File berhasil dimuat!")
    display(cases.head())
else:
    print(f"File tidak ditemukan di: {csv_path}")
    print("Pastikan file 'cases.csv' sudah ada di folder tersebut setelah proses Tahap 1 & 2.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File berhasil dimuat!


,case_id,file_name,nomor_perkara,tahun,tingkat_peradilan,pengadilan_asal,jenis_kdrt,pasal,terdakwa,korban,ringkasan_fakta,argumen_hukum,amar_putusan,outcome,retrieval_text,word_count,unique_words
0,1,case_001.txt,UNKNOWN,2014,Kasasi,tanjungkara ng sejak tanggal h k,Fisik,pasal 44 ayat 1 undang undang nomor 23 tahun 2...,n a nama sandy ringgala s e bin i syamsul bahr...,duduk di a pinggir kasur dan anak saksi devpi ...,a i s e n o d n i k i l b u p e a r i s g e n ...,UNKNOWN,mengadili perkaranya dengan sengaja melakukan ...,unknown,Fisik pasal 44 ayat 1 undang undang nomor 23 t...,5719,816
1,2,case_002.txt,UNKNOWN,2023,Kasasi,UNKNOWN,Fisik,pasal 44 ayat 2 undang undang nomor 23 tahun 2...,i h nama triamana hadkianta a pangkat nrp sertu,mendapat jatuh sakit sebagaimana diatur dan s ...,a i s e n o d n i k i l b u p e a r i s g e n ...,UNKNOWN,UNKNOWN,unknown,Fisik pasal 44 ayat 2 undang undang nomor 23 t...,3895,551
2,3,case_003.txt,UNKNOWN,2010,Kasasi,mungkid karena n n didakwa buahwa terdakwa nur...,Penelantaran,pasal 34 ayat 1 undang undang e republik indon...,n nama nuryanto bin sunarto slamet i tempat la...,yang akhirnya e jumini mengandung dan melahirk...,a i s e n o d n i k i l b u p e a r i s g e n ...,UNKNOWN,UNKNOWN,unknown,Penelantaran pasal 34 ayat 1 undang undang e r...,3583,644
3,4,case_004.txt,UNKNOWN,2013,Kasasi,pekanbaru karena o u didakwa d g pertama n aba...,Fisik,pasal 44 ayat 4 undang unedang ri nomor 23 tah...,i n a m a said zainal bin said umar usman h k ...,wan maryam alias mar yang merupakan istri s m ...,a i s e n o d n i k i l b u p e a r i s g e n ...,menimbang bahwa putusan pengadilan tinggi ters...,mengadili tidak dilaksanakan menurut ketentuan...,kasasi_ditolak,Fisik pasal 44 ayat 4 undang unedang ri nomor ...,4610,693
4,5,case_005.txt,UNKNOWN,2018,Kasasi,padang o u karena didakwa dengan dakwaan tungg...,Penelantaran,pasal 49 huruf a undang n aundang nomor 23 tah...,telah memutus perkara terdakwa nama devia idru...,imerupakan suami h istri sesuai akta nikah nomor,a i s e n o d n i k i l b u p e a r i s g e n ...,menimbang bahwa putusan pengadilan tinggi pada...,mengadili tidak s m dilaksanakan gmenurut kete...,kasasi_ditolak,Penelantaran pasal 49 huruf a undang n aundang...,3210,528


In [ ]:
from sklearn.model_selection import train_test_split
train_df,test_df=train_test_split(cases,test_size=0.2,random_state=42,stratify=None)

## TF-IDF Representation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=10000,ngram_range=(1,2))
X_train=vectorizer.fit_transform(train_df['retrieval_text'])
X_test=vectorizer.transform(test_df['retrieval_text'])

## TF-IDF + Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_tfidf(query,k=5):
 q=vectorizer.transform([query])
 scores=cosine_similarity(q,X_train)[0]
 idx=scores.argsort()[::-1][:k]
 return train_df.iloc[idx][['case_id','outcome']].assign(similarity=scores[idx])

## TF-IDF + SVM

In [ ]:
from sklearn.svm import LinearSVC
svm_model=LinearSVC()
svm_model.fit(X_train,train_df['outcome'])
svm_pred=svm_model.predict(X_test)

## TF-IDF + Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb_model=MultinomialNB()
nb_model.fit(X_train,train_df['outcome'])
nb_pred=nb_model.predict(X_test)

## IndoBERT Embedding

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModel
MODEL='indobenchmark/indobert-base-p1'
tokenizer=AutoTokenizer.from_pretrained(MODEL)
model=AutoModel.from_pretrained(MODEL)

def bert_embedding(text):
 inp=tokenizer(str(text),return_tensors='pt',truncation=True,max_length=512,padding=True)
 with torch.no_grad():
  out=model(**inp)
 return out.last_hidden_state.mean(dim=1).squeeze().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
bert_vectors=np.array([bert_embedding(x) for x in train_df['retrieval_text']])

In [ ]:
def retrieve_bert(query,k=5):
 q=bert_embedding(query)
 scores=cosine_similarity([q],bert_vectors)[0]
 idx=scores.argsort()[::-1][:k]
 return train_df.iloc[idx][['case_id','outcome']].assign(similarity=scores[idx])

## Fungsi retrieve() Utama

In [ ]:
def retrieve(query,k=5,method='tfidf'):
 if method=='tfidf':
  return retrieve_tfidf(query,k)
 elif method=='bert':
  return retrieve_bert(query,k)
 else:
  raise ValueError('method: tfidf atau bert')

## Evaluasi Precision@K dan Recall@K

In [ ]:
def precision_at_k(retrieved,relevant,k):
 return len(set(retrieved[:k]) & set(relevant))/k

def recall_at_k(retrieved,relevant,k):
 return len(set(retrieved[:k]) & set(relevant))/len(relevant) if len(relevant)>0 else 0

## Query Evaluation Dataset

In [ ]:
os.makedirs('data/eval',exist_ok=True)
queries=[]
for _,row in cases.head(min(10,len(cases))).iterrows():
 queries.append({'query':str(row['ringkasan_fakta'])[:200],'ground_truth_case_id':int(row['case_id'])})
with open('data/eval/queries.json','w',encoding='utf-8') as f:
 json.dump(queries,f,ensure_ascii=False,indent=2)

## Evaluasi Model Klasifikasi

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
print('SVM Accuracy:',accuracy_score(test_df['outcome'],svm_pred))
print('NB Accuracy:',accuracy_score(test_df['outcome'],nb_pred))

SVM Accuracy: 0.5
NB Accuracy: 0.5


### Menyimpan Output Tahap 3 ke Google Drive
Kode di bawah ini akan memastikan file `queries.json` yang telah Anda buat tersimpan permanen di folder Google Drive Anda.

In [ ]:
import shutil

# Tentukan path tujuan di Drive
drive_eval_path = os.path.join(PDF_FOLDER, 'data/eval')
os.makedirs(drive_eval_path, exist_ok=True)

# Salin file dari lokal Colab ke Drive
source_file = 'data/eval/queries.json'
dest_file = os.path.join(drive_eval_path, 'queries.json')

if os.path.exists(source_file):
    shutil.copy(source_file, dest_file)
    print(f'Berhasil! File evaluasi telah disimpan secara permanen di: {dest_file}')
else:
    print('File sumber tidak ditemukan. Pastikan cell pembuatan queries.json sudah dijalankan.')

Berhasil! File evaluasi telah disimpan secara permanen di: /content/drive/My Drive/TugasPK/Test/raw_pdf/data/eval/queries.json


### Opsional: Menyimpan Model dan Embeddings ke Drive
Agar tidak perlu melatih ulang model di masa depan, kita simpan model SVM, Naive Bayes, dan hasil embedding IndoBERT.

In [ ]:
import joblib
import pickle

# Tentukan folder model di Drive
model_path = os.path.join(PDF_FOLDER, 'models')
os.makedirs(model_path, exist_ok=True)

# 1. Simpan Vectorizer TF-IDF dan Model ML
joblib.dump(vectorizer, os.path.join(model_path, 'tfidf_vectorizer.pkl'))
joblib.dump(svm_model, os.path.join(model_path, 'svm_model.pkl'))
joblib.dump(nb_model, os.path.join(model_path, 'nb_model.pkl'))

# 2. Simpan BERT Embeddings (karena prosesnya lama)
with open(os.path.join(model_path, 'bert_vectors.pkl'), 'wb') as f:
    pickle.dump(bert_vectors, f)

print(f"Semua model dan embedding berhasil disimpan di: {model_path}")

Semua model dan embedding berhasil disimpan di: /content/drive/My Drive/TugasPK/Test/raw_pdf/models


### Contoh Cara Menggunakan Fungsi Retrieval
Anda bisa mencoba memanggil fungsi ini untuk melihat hasil kemiripan kasus:

In [ ]:
# Contoh query kasus baru
example_query = "Kasus kekerasan dalam rumah tangga fisik pasal 5"

print("--- Hasil Retrieval TF-IDF ---")
display(retrieve(example_query, k=3, method='tfidf'))

print("\n--- Hasil Retrieval IndoBERT ---")
display(retrieve(example_query, k=3, method='bert'))

--- Hasil Retrieval TF-IDF ---


,case_id,outcome,similarity
19,20,unknown,0.216583
1,2,unknown,0.212748
25,26,unknown,0.177950



--- Hasil Retrieval IndoBERT ---


,case_id,outcome,similarity
19,20,unknown,0.433234
1,2,unknown,0.426899
11,12,unknown,0.422795


## Output Tahap 3

- retrieve()
- TF-IDF + Cosine
- TF-IDF + SVM
- TF-IDF + Naive Bayes
- IndoBERT + Cosine
- Precision@K
- Recall@K
- data/eval/queries.json